# Baseline vs Transfer Performance on Untouched Holdout

This notebook compares:
- Baseline ensemble predictions
- Transfer (local_only) ensemble predictions
- Transfer (ablation) ensemble predictions

All metrics in the main tables are computed **only** on the untouched holdout IDs from `processed_data/local/final_holdout/final_holdout_ids.csv`.

In [3]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

ROOT = Path('.').resolve()

PATHS = {
    'holdout_ids': ROOT / 'processed_data' / 'local' / 'final_holdout' / 'final_holdout_ids.csv',
    'local_all_raw': ROOT / 'processed_data' / 'local' / 'local_all_raw.csv',

    'baseline_holdout': ROOT / 'predictions' / 'local' / 'local_baseline_holdout_predictions.csv',
    'baseline_primary': ROOT / 'predictions' / 'local' / 'local_baseline_predictions.csv',
    'baseline_alt': ROOT / 'predictions' / 'ensemble_predictions.csv',

    'transfer_local_only_holdout': ROOT / 'predictions' / 'transfer' / 'local_only' / 'ensemble_predictions_holdout_transfer.csv',
    'transfer_local_only': ROOT / 'predictions' / 'transfer' / 'local_only' / 'ensemble_predictions_transfer.csv',

    'transfer_ablation_holdout': ROOT / 'predictions' / 'transfer' / 'ablation' / 'ensemble_predictions_holdout_transfer.csv',
    'transfer_ablation': ROOT / 'predictions' / 'transfer' / 'ablation' / 'ensemble_predictions_transfer.csv',
}

for k, v in PATHS.items():
    print(f'{k:28s} -> {v} | exists={v.exists()}')

holdout_ids                  -> C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\final_holdout\final_holdout_ids.csv | exists=True
local_all_raw                -> C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\local_all_raw.csv | exists=True
baseline_holdout             -> C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\local\local_baseline_holdout_predictions.csv | exists=False
baseline_primary             -> C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\local\local_baseline_predictions.csv | exists=True
baseline_alt                 -> C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\ensemble_predictions.csv | exists=True
transfer_local_only_holdout  -> C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\transfer\local_only\ensemble_predictions_holdout_transfer.csv | exists=False
transfer_local_only          -> C:\Users\micof\Document

In [17]:
def choose_column(df, candidates, required=False):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f'Missing required column. Tried: {candidates}')
    return None

def load_experiment(name, path, holdout_df):
    if not path.exists():
        return None

    df = pd.read_csv(path, dtype={'product_id': str})
    if 'product_id' not in df.columns:
        raise KeyError(f'{name}: missing product_id column in {path}')

    score_col = choose_column(
        df,
        [
            'ensemble_fraud_proba',
            'ensemble_tuned_proba',
            'ensemble_weighted_auc_proba',
            'ensemble_avg_proba',
        ],
        required=True,
    )
    pred_col = choose_column(df, ['fraud_prediction', 'ensemble_final_pred'])
    label_col = choose_column(df, ['fraud_label', 'actual_label'])

    modality_cols = [
        c for c in ['text_fraud_proba', 'image_fraud_proba', 'metadata_fraud_proba']
        if c in df.columns
    ]

    keep_cols = ['product_id', score_col] + modality_cols
    if pred_col:
        keep_cols.append(pred_col)
    if label_col:
        keep_cols.append(label_col)

    slim = df[keep_cols].copy()
    slim = slim.rename(columns={score_col: 'score'})

    if pred_col:
        slim = slim.rename(columns={pred_col: 'pred'})
    else:
        slim['pred'] = (slim['score'] >= 0.5).astype(int)

    if label_col:
        slim = slim.rename(columns={label_col: 'label_from_file'})

    merged = holdout_df.merge(slim, on='product_id', how='left')
    merged['experiment'] = name

    if 'label_from_file' in merged.columns:
        valid_lbl = merged['label_from_file'].notna()
        mismatch = (merged.loc[valid_lbl, 'fraud_label'].astype(int) != merged.loc[valid_lbl, 'label_from_file'].astype(int)).sum()
    else:
        mismatch = 0

    return {
        'name': name,
        'path': str(path),
        'df': merged,
        'rows_total_in_file': len(df),
        'rows_with_holdout_overlap': merged['score'].notna().sum(),
        'label_mismatch_vs_holdout': int(mismatch),
    }

def calc_metrics(y_true, y_pred, y_score):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    out = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else np.nan,
        'avg_precision': average_precision_score(y_true, y_score) if len(np.unique(y_true)) > 1 else np.nan,
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn),
        'predicted_fraud': int(y_pred.sum()),
    }
    return out

def file_holdout_overlap(path, holdout_ids_set):
    if not path.exists():
        return -1
    try:
        tmp = pd.read_csv(path, usecols=['product_id'], dtype={'product_id': str})
    except Exception:
        return -1
    return int(tmp['product_id'].isin(holdout_ids_set).sum())

holdout = pd.read_csv(PATHS['holdout_ids'], dtype={'product_id': str})
holdout['fraud_label'] = holdout['fraud_label'].astype(int)

# Bring in source/platform tag for adaptability analysis.
if PATHS['local_all_raw'].exists():
    source_df = pd.read_csv(PATHS['local_all_raw'], dtype={'product_id': str})
    if 'source' in source_df.columns:
        holdout = holdout.merge(
            source_df[['product_id', 'source']].drop_duplicates('product_id'),
            on='product_id',
            how='left',
        )

holdout_ids_set = set(holdout['product_id'].astype(str))

candidate_map = {
    'baseline': [PATHS['baseline_holdout'], PATHS['baseline_primary'], PATHS['baseline_alt']],
    'transfer_local_only': [PATHS['transfer_local_only_holdout'], PATHS['transfer_local_only']],
    'transfer_ablation': [PATHS['transfer_ablation_holdout'], PATHS['transfer_ablation']],
}

selected_paths = {}
selection_rows = []

for name, candidates in candidate_map.items():
    best_path = None
    best_overlap = -1
    for p in candidates:
        ov = file_holdout_overlap(p, holdout_ids_set)
        if ov > best_overlap:
            best_overlap = ov
            best_path = p
    selected_paths[name] = best_path
    selection_rows.append({
        'experiment': name,
        'selected_path': str(best_path),
        'selected_overlap': int(best_overlap),
    })

selection_df = pd.DataFrame(selection_rows).sort_values('experiment').reset_index(drop=True)
print('Selected prediction files (max overlap with untouched holdout):')
display(selection_df)

if (selection_df['selected_overlap'] <= 0).all():
    print('WARNING: No selected file overlaps with untouched holdout IDs.')
    print('Run the generation cell below to produce holdout-specific predictions, then rerun this cell.')

experiments = []
for name, p in selected_paths.items():
    ex = load_experiment(name, p, holdout)
    if ex is not None:
        experiments.append(ex)

if not experiments:
    raise RuntimeError('No experiment files were found. Check PATHS in Cell 2.')

coverage_rows = []
for ex in experiments:
    coverage_rows.append({
        'experiment': ex['name'],
        'source_file': ex['path'],
        'rows_in_file': ex['rows_total_in_file'],
        'holdout_overlap': ex['rows_with_holdout_overlap'],
        'label_mismatch_vs_holdout': ex['label_mismatch_vs_holdout'],
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values('experiment').reset_index(drop=True)
coverage_df

Selected prediction files (max overlap with untouched holdout):


,experiment,selected_path,selected_overlap
0,baseline,C:\Users\micof\Documents\GitHub\e-commerce-fra...,70
1,transfer_ablation,C:\Users\micof\Documents\GitHub\e-commerce-fra...,70
2,transfer_local_only,C:\Users\micof\Documents\GitHub\e-commerce-fra...,70


,experiment,source_file,rows_in_file,holdout_overlap,label_mismatch_vs_holdout
0,baseline,C:\Users\micof\Documents\GitHub\e-commerce-fra...,70,70,0
1,transfer_ablation,C:\Users\micof\Documents\GitHub\e-commerce-fra...,70,70,0
2,transfer_local_only,C:\Users\micof\Documents\GitHub\e-commerce-fra...,70,70,0


## Optional: Generate Holdout-Specific Predictions

Use this section if overlap is zero in the selection table above.
It will score the untouched holdout IDs directly using baseline and transfer model profiles, then save holdout prediction files used by this notebook.

In [ ]:
AUTO_GENERATE_HOLDOUT_PREDICTIONS = False  # Set to True only when you need to regenerate files

if AUTO_GENERATE_HOLDOUT_PREDICTIONS:
    if not PATHS['local_all_raw'].exists():
        raise FileNotFoundError(f'Missing raw local dataset: {PATHS["local_all_raw"]}')

    raw_df = pd.read_csv(PATHS['local_all_raw'], dtype={'product_id': str})
    holdout_raw = raw_df[raw_df['product_id'].isin(holdout['product_id'])].copy()

    holdout_input_path = ROOT / 'processed_data' / 'local' / 'final_holdout' / 'local_final_holdout_raw_for_predict.csv'
    holdout_raw.to_csv(holdout_input_path, index=False)
    print(f'Prepared holdout inference input: {holdout_input_path} ({len(holdout_raw)} rows)')

    commands = [
        [
            sys.executable, 'predict.py',
            '--profile', 'global',
            '--input', str(holdout_input_path),
            '--output', str(PATHS['baseline_holdout']),
        ],
        [
            sys.executable, 'predict.py',
            '--profile', 'local_only',
            '--input', str(holdout_input_path),
            '--output', str(PATHS['transfer_local_only_holdout']),
        ],
        [
            sys.executable, 'predict.py',
            '--profile', 'ablation',
            '--input', str(holdout_input_path),
            '--output', str(PATHS['transfer_ablation_holdout']),
        ],
    ]

    for cmd in commands:
        print('Running:', ' '.join(cmd))
        result = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
        print(result.stdout[-2000:])
        if result.returncode != 0:
            print(result.stderr[-2000:])
            raise RuntimeError(f'Command failed with code {result.returncode}: {cmd}')

    print('Holdout predictions generated. Rerun Cell 3 to refresh selected file overlaps.')
else:
    print('Auto-generation disabled. Set AUTO_GENERATE_HOLDOUT_PREDICTIONS=True only when you want to regenerate holdout predictions.')

Prepared holdout inference input: C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\final_holdout\local_final_holdout_raw_for_predict.csv (70 rows)
Running: c:\Users\micof\anaconda3\python.exe predict.py --profile global --input C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\processed_data\local\final_holdout\local_final_holdout_raw_for_predict.csv --output C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\local\local_baseline_holdout_predictions.csv
model:    0.3259
    Metadata model: 0.0030
    Ensemble:       0.2599
    Actual:         LEGITIMATE (CORRECT)

[OK] Product 900317: LEGITIMATE (confidence: 57.0%)
    Text model:     0.4669
    Image model:    0.1806
    Metadata model: 0.0050
    Ensemble:       0.2148
    Actual:         LEGITIMATE (CORRECT)

[OK] Product 900324: LEGITIMATE (confidence: 59.9%)
    Text model:     0.4672
    Image model:    0.1352
    Metadata model: 0.0049
    Ensemble:       0.2004
    Act

In [18]:
metric_rows = []
per_exp_scored = {}

for ex in experiments:
    scored = ex['df'][ex['df']['score'].notna()].copy()
    if scored.empty:
        print(f"Skipping {ex['name']} (no overlap rows with scores).")
        continue

    scored['pred'] = scored['pred'].astype(int)
    scored['fraud_label'] = scored['fraud_label'].astype(int)

    y_true = scored['fraud_label'].values
    y_pred = scored['pred'].values
    y_score = scored['score'].values

    m = calc_metrics(y_true, y_pred, y_score)
    m['experiment'] = ex['name']
    m['n_eval'] = len(scored)
    metric_rows.append(m)

    per_exp_scored[ex['name']] = scored

metrics_df = pd.DataFrame(metric_rows)
if not metrics_df.empty:
    metrics_df = metrics_df.sort_values('experiment').reset_index(drop=True)

display_cols = [
    'experiment', 'n_eval', 'accuracy', 'f1', 'precision', 'recall',
    'roc_auc', 'avg_precision', 'tp', 'fp', 'tn', 'fn', 'predicted_fraud'
]

if metrics_df.empty:
    print('No scorable rows found for untouched holdout IDs.')
    print('Run the holdout generation cell (Cell 5), then rerun Cell 3 and onward.')
else:
    display(
        metrics_df[display_cols].style.format({
            'accuracy': '{:.4f}',
            'f1': '{:.4f}',
            'precision': '{:.4f}',
            'recall': '{:.4f}',
            'roc_auc': '{:.4f}',
            'avg_precision': '{:.4f}',
        })
    )

,experiment,n_eval,accuracy,f1,precision,recall,roc_auc,avg_precision,tp,fp,tn,fn,predicted_fraud
0,baseline,70,0.7857,0.2105,0.3333,0.1538,0.6964,0.3843,2,4,53,11,6
1,transfer_ablation,70,0.8429,0.6857,0.5455,0.9231,0.9163,0.8381,12,10,47,1,22
2,transfer_local_only,70,0.8714,0.7273,0.6000,0.9231,0.9170,0.8406,12,8,49,1,20


## Highlighted Headline Results

This compact view is for presentation/reporting.
It directly contrasts the previous baseline model versus the best transfer model on untouched holdout IDs.

In [22]:
if metrics_df.empty:
    print('Highlighted results unavailable until holdout metrics are computed.')
else:
    baseline_row = metrics_df.loc[metrics_df['experiment'] == 'baseline']
    transfer_rows = metrics_df[metrics_df['experiment'].str.contains('transfer', na=False)]

    if baseline_row.empty or transfer_rows.empty:
        print('Need both baseline and transfer rows for headline comparison.')
    else:
        baseline_row = baseline_row.iloc[0]
        best_transfer = transfer_rows.sort_values('f1', ascending=False).iloc[0]

        highlight = pd.DataFrame([
            {
                'model': 'baseline_previous',
                'accuracy': baseline_row['accuracy'],
                'precision': baseline_row['precision'],
                'recall': baseline_row['recall'],
                'f1': baseline_row['f1'],
                'roc_auc': baseline_row['roc_auc'],
                'avg_precision': baseline_row['avg_precision'],
            },
            {
                'model': f"proposed_{best_transfer['experiment']}",
                'accuracy': best_transfer['accuracy'],
                'precision': best_transfer['precision'],
                'recall': best_transfer['recall'],
                'f1': best_transfer['f1'],
                'roc_auc': best_transfer['roc_auc'],
                'avg_precision': best_transfer['avg_precision'],
            },
        ])

        delta = pd.DataFrame([
            {
                'comparison': f"{best_transfer['experiment']} - baseline",
                'accuracy_delta': best_transfer['accuracy'] - baseline_row['accuracy'],
                'precision_delta': best_transfer['precision'] - baseline_row['precision'],
                'recall_delta': best_transfer['recall'] - baseline_row['recall'],
                'f1_delta': best_transfer['f1'] - baseline_row['f1'],
                'roc_auc_delta': best_transfer['roc_auc'] - baseline_row['roc_auc'],
                'avg_precision_delta': best_transfer['avg_precision'] - baseline_row['avg_precision'],
            }
        ])

        print('Headline comparison on untouched holdout:')
        display(
            highlight.style.format({
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
                'avg_precision': '{:.4f}',
            })
        )

        print('Improvement of proposed transfer model over baseline:')
        display(
            delta.style.format({
                'accuracy_delta': '{:+.4f}',
                'precision_delta': '{:+.4f}',
                'recall_delta': '{:+.4f}',
                'f1_delta': '{:+.4f}',
                'roc_auc_delta': '{:+.4f}',
                'avg_precision_delta': '{:+.4f}',
            })
        )

Headline comparison on untouched holdout:


,model,accuracy,precision,recall,f1,roc_auc,avg_precision
0,baseline_previous,0.7857,0.3333,0.1538,0.2105,0.6964,0.3843
1,proposed_transfer_local_only,0.8714,0.6000,0.9231,0.7273,0.9170,0.8406


Improvement of proposed transfer model over baseline:


,comparison,accuracy_delta,precision_delta,recall_delta,f1_delta,roc_auc_delta,avg_precision_delta
0,transfer_local_only - baseline,+0.0857,+0.2667,+0.7692,+0.5167,+0.2206,+0.4563


## Baseline Single Modalities vs Best Transfer Ensemble

This section compares baseline unimodal models (text, image, metadata) against the strongest transfer-learning ensemble on untouched holdout IDs.
It highlights absolute metrics and the delta of the best transfer ensemble relative to each baseline modality.

In [25]:
if metrics_df.empty or 'unimodal_metrics_df' not in globals() or unimodal_metrics_df.empty:
    print('Comparison unavailable: run the metrics and unimodal cells first.')
else:
    baseline_uni = unimodal_metrics_df[unimodal_metrics_df['experiment'] == 'baseline'].copy()
    transfer_ens = metrics_df[metrics_df['experiment'].str.contains('transfer', na=False)].copy()

    if baseline_uni.empty or transfer_ens.empty:
        print('Comparison unavailable: baseline unimodal rows or transfer ensemble rows are missing.')
    else:
        best_transfer = transfer_ens.sort_values('f1', ascending=False).iloc[0]

        base_cols = ['modality', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision']
        baseline_view = baseline_uni[base_cols].copy()
        baseline_view = baseline_view.sort_values('modality').reset_index(drop=True)

        transfer_row = pd.DataFrame([
            {
                'modality': f"BEST_TRANSFER_ENSEMBLE ({best_transfer['experiment']})",
                'accuracy': best_transfer['accuracy'],
                'precision': best_transfer['precision'],
                'recall': best_transfer['recall'],
                'f1': best_transfer['f1'],
                'roc_auc': best_transfer['roc_auc'],
                'avg_precision': best_transfer['avg_precision'],
            }
        ])

        combined_compare = pd.concat([baseline_view, transfer_row], ignore_index=True)
        print('Baseline single modalities vs best transfer ensemble (untouched holdout):')
        display(
            combined_compare.style.format({
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
                'avg_precision': '{:.4f}',
            })
        )

        delta_rows = []
        for _, r in baseline_view.iterrows():
            delta_rows.append({
                'vs_baseline_modality': r['modality'],
                'accuracy_delta': best_transfer['accuracy'] - r['accuracy'],
                'precision_delta': best_transfer['precision'] - r['precision'],
                'recall_delta': best_transfer['recall'] - r['recall'],
                'f1_delta': best_transfer['f1'] - r['f1'],
                'roc_auc_delta': best_transfer['roc_auc'] - r['roc_auc'],
                'avg_precision_delta': best_transfer['avg_precision'] - r['avg_precision'],
            })

        delta_df = pd.DataFrame(delta_rows)
        print(f"Improvement of best transfer ensemble ({best_transfer['experiment']}) vs each baseline modality:")
        display(
            delta_df.style.format({
                'accuracy_delta': '{:+.4f}',
                'precision_delta': '{:+.4f}',
                'recall_delta': '{:+.4f}',
                'f1_delta': '{:+.4f}',
                'roc_auc_delta': '{:+.4f}',
                'avg_precision_delta': '{:+.4f}',
            })
        )

Baseline single modalities vs best transfer ensemble (untouched holdout):


,modality,accuracy,precision,recall,f1,roc_auc,avg_precision
0,image,0.7429,0.1429,0.0769,0.1000,0.4507,0.1938
1,metadata,0.8429,1.0000,0.1538,0.2667,0.9231,0.8323
2,text,0.8143,0.0000,0.0000,0.0000,0.3576,0.1585
3,BEST_TRANSFER_ENSEMBLE (transfer_local_only),0.8714,0.6000,0.9231,0.7273,0.9170,0.8406


Improvement of best transfer ensemble (transfer_local_only) vs each baseline modality:


,vs_baseline_modality,accuracy_delta,precision_delta,recall_delta,f1_delta,roc_auc_delta,avg_precision_delta
0,image,+0.1286,+0.4571,+0.8462,+0.6273,+0.4663,+0.6468
1,metadata,+0.0286,-0.4000,+0.7692,+0.4606,-0.0061,+0.0082
2,text,+0.0571,+0.6000,+0.9231,+0.7273,+0.5594,+0.6821


## Objective Alignment: Unimodal vs Multimodal Gains

This section directly addresses the specific objective on improving detection over unimodal and multimodal baselines.

- Unimodal metrics are computed from each experiment file's modality probabilities.
- Multimodal metric is the ensemble score used by that experiment file.
- The uplift table shows how much the multimodal model improves over the best unimodal model in the same experiment.

In [19]:
if metrics_df.empty:
    print('Unimodal/ensemble uplift is unavailable until holdout-scored predictions are loaded.')
else:
    def modality_unimodal_metrics(scored_df):
        rows = []
        y_true = scored_df['fraud_label'].astype(int).values

        modality_map = {
            'text': 'text_fraud_proba',
            'image': 'image_fraud_proba',
            'metadata': 'metadata_fraud_proba',
        }

        for modality, col in modality_map.items():
            if col not in scored_df.columns:
                continue
            tmp = scored_df[scored_df[col].notna()].copy()
            if tmp.empty:
                continue
            y_t = tmp['fraud_label'].astype(int).values
            y_s = tmp[col].astype(float).values
            y_p = (y_s >= 0.5).astype(int)
            m = calc_metrics(y_t, y_p, y_s)
            m['modality'] = modality
            m['n_eval'] = len(tmp)
            rows.append(m)

        return pd.DataFrame(rows)

    unimodal_rows = []
    uplift_rows = []

    for ex_name, scored in per_exp_scored.items():
        uni_df = modality_unimodal_metrics(scored)
        if uni_df.empty:
            continue

        best_uni = uni_df.sort_values('f1', ascending=False).iloc[0]
        ens = metrics_df.loc[metrics_df['experiment'] == ex_name].iloc[0]

        for _, r in uni_df.iterrows():
            rec = r.to_dict()
            rec['experiment'] = ex_name
            unimodal_rows.append(rec)

        uplift_rows.append({
            'experiment': ex_name,
            'best_unimodal_modality_by_f1': best_uni['modality'],
            'best_unimodal_f1': float(best_uni['f1']),
            'ensemble_f1': float(ens['f1']),
            'f1_uplift_vs_best_unimodal': float(ens['f1'] - best_uni['f1']),
            'best_unimodal_recall': float(best_uni['recall']),
            'ensemble_recall': float(ens['recall']),
            'recall_uplift_vs_best_unimodal': float(ens['recall'] - best_uni['recall']),
        })

    unimodal_metrics_df = pd.DataFrame(unimodal_rows)
    objective1_uplift_df = pd.DataFrame(uplift_rows).sort_values('experiment').reset_index(drop=True)

    print('Unimodal metrics (holdout):')
    if not unimodal_metrics_df.empty:
        display(
            unimodal_metrics_df[['experiment', 'modality', 'n_eval', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision']]
            .sort_values(['experiment', 'modality'])
            .style.format({
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
                'avg_precision': '{:.4f}',
            })
        )
    else:
        print('No unimodal probability columns found in selected files.')

    print('Ensemble uplift vs best unimodal (Objective 1 evidence):')
    if not objective1_uplift_df.empty:
        display(
            objective1_uplift_df.style.format({
                'best_unimodal_f1': '{:.4f}',
                'ensemble_f1': '{:.4f}',
                'f1_uplift_vs_best_unimodal': '{:+.4f}',
                'best_unimodal_recall': '{:.4f}',
                'ensemble_recall': '{:.4f}',
                'recall_uplift_vs_best_unimodal': '{:+.4f}',
            })
        )
    else:
        print('Uplift table unavailable (insufficient modality data).')

Unimodal metrics (holdout):


,experiment,modality,n_eval,accuracy,precision,recall,f1,roc_auc,avg_precision
1,baseline,image,70,0.7429,0.1429,0.0769,0.1000,0.4507,0.1938
2,baseline,metadata,70,0.8429,1.0000,0.1538,0.2667,0.9231,0.8323
0,baseline,text,70,0.8143,0.0000,0.0000,0.0000,0.3576,0.1585
7,transfer_ablation,image,70,0.5857,0.2647,0.6923,0.3830,0.6613,0.3889
8,transfer_ablation,metadata,70,0.9429,0.8000,0.9231,0.8571,0.9217,0.8126
6,transfer_ablation,text,70,0.8143,0.0000,0.0000,0.0000,0.6559,0.4024
4,transfer_local_only,image,70,0.3143,0.2034,0.9231,0.3333,0.5452,0.3365
5,transfer_local_only,metadata,70,0.9000,0.7500,0.6923,0.7200,0.9271,0.8362
3,transfer_local_only,text,70,0.8143,0.0000,0.0000,0.0000,0.6727,0.4803


Ensemble uplift vs best unimodal (Objective 1 evidence):


,experiment,best_unimodal_modality_by_f1,best_unimodal_f1,ensemble_f1,f1_uplift_vs_best_unimodal,best_unimodal_recall,ensemble_recall,recall_uplift_vs_best_unimodal
0,baseline,metadata,0.2667,0.2105,-0.0561,0.1538,0.1538,+0.0000
1,transfer_ablation,metadata,0.8571,0.6857,-0.1714,0.9231,0.9231,+0.0000
2,transfer_local_only,metadata,0.7200,0.7273,+0.0073,0.6923,0.9231,+0.2308


## Objective Alignment: Cross-Platform Adaptability (By Source)

This section evaluates adaptability using the available source/domain tag.

If source values are available for holdout IDs, metrics are broken down per source for baseline and transfer models.
If no source field is available, the cell will report that and skip this analysis.

In [20]:
if metrics_df.empty:
    print('Source-level adaptability cannot be computed until holdout-scored predictions are available.')
    source_metrics_df = pd.DataFrame()
else:
    source_metrics_rows = []

    for ex_name, scored in per_exp_scored.items():
        if 'source' not in scored.columns:
            continue

        for src, grp in scored.groupby('source', dropna=False):
            if len(grp) < 3:
                continue
            y_true = grp['fraud_label'].astype(int).values
            if len(np.unique(y_true)) < 2:
                continue
            y_score = grp['score'].astype(float).values
            y_pred = grp['pred'].astype(int).values
            m = calc_metrics(y_true, y_pred, y_score)
            m['experiment'] = ex_name
            m['source'] = 'unknown' if pd.isna(src) else str(src)
            m['n_eval'] = len(grp)
            source_metrics_rows.append(m)

    source_metrics_df = pd.DataFrame(source_metrics_rows)

    if source_metrics_df.empty:
        print('No source-level adaptability breakdown available (missing source tags or insufficient class variety per source).')
    else:
        src_disp = source_metrics_df[['experiment', 'source', 'n_eval', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision']]
        display(
            src_disp.sort_values(['source', 'experiment']).style.format({
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
                'avg_precision': '{:.4f}',
            })
        )

        pivot_f1 = source_metrics_df.pivot_table(index='source', columns='experiment', values='f1', aggfunc='mean')
        print('F1 by source and experiment:')
        display(pivot_f1.style.format('{:.4f}'))

,experiment,source,n_eval,accuracy,precision,recall,f1,roc_auc,avg_precision
0,baseline,local1,67,0.7910,0.0000,0.0000,0.0000,0.6175,0.2049
2,transfer_ablation,local1,67,0.8358,0.4737,0.9000,0.6207,0.8982,0.8033
1,transfer_local_only,local1,67,0.8657,0.5294,0.9000,0.6667,0.9079,0.8531


F1 by source and experiment:


experiment,baseline,transfer_ablation,transfer_local_only
source,,,
local1,0.0000,0.6207,0.6667


## Objective Alignment Summary (Auto-Generated)

This cell generates a concise narrative that you can directly use in your write-up.
It links observed results to each specific objective using the computed holdout metrics.

In [21]:
if metrics_df.empty:
    print('Objective narrative will be generated after holdout-scored predictions are available.')
else:
    def safe_get(df, row_key, col):
        if df.empty or row_key not in set(df['experiment']):
            return np.nan
        return float(df.loc[df['experiment'] == row_key, col].iloc[0])

    exp_names = list(metrics_df['experiment'])
    best_by_f1 = metrics_df.sort_values('f1', ascending=False).iloc[0]

    baseline_name = 'baseline' if 'baseline' in exp_names else None
    best_transfer_name = None
    transfer_only_df = metrics_df[metrics_df['experiment'].str.contains('transfer', na=False)]
    if not transfer_only_df.empty:
        best_transfer_name = transfer_only_df.sort_values('f1', ascending=False).iloc[0]['experiment']

    print('General Objective:')
    print('Develop and evaluate a multimodal fraud detection framework on local untouched holdout data.')
    print(f"Best evaluated multimodal setting by F1: {best_by_f1['experiment']} (F1={best_by_f1['f1']:.4f}, Recall={best_by_f1['recall']:.4f}, Precision={best_by_f1['precision']:.4f}, Accuracy={best_by_f1['accuracy']:.4f})")
    print()

    print('Specific Objective 1 (improvement vs unimodal/baseline):')
    if 'objective1_uplift_df' in globals() and not objective1_uplift_df.empty:
        for _, r in objective1_uplift_df.iterrows():
            print(
                f"- {r['experiment']}: ensemble F1 {r['ensemble_f1']:.4f} vs best unimodal ({r['best_unimodal_modality_by_f1']}) F1 {r['best_unimodal_f1']:.4f}; "
                f"uplift={r['f1_uplift_vs_best_unimodal']:+.4f}, recall uplift={r['recall_uplift_vs_best_unimodal']:+.4f}"
            )
    else:
        print('- Unimodal uplift table unavailable (missing modality probability columns).')
    print()

    print('Specific Objective 2 (adaptation to actual local datasets):')
    print(f"- Holdout IDs evaluated: {len(holdout)}")
    print('- Experiments loaded from actual run outputs:')
    for _, row in coverage_df.iterrows():
        print(f"  - {row['experiment']}: overlap={int(row['holdout_overlap'])}, label_mismatch={int(row['label_mismatch_vs_holdout'])}, file={row['source_file']}")
    print()

    print('Specific Objective 3 (deep learning multimodal improvement and cross-platform adaptability):')
    if baseline_name and best_transfer_name:
        b_f1 = safe_get(metrics_df, baseline_name, 'f1')
        t_f1 = safe_get(metrics_df, best_transfer_name, 'f1')
        b_rec = safe_get(metrics_df, baseline_name, 'recall')
        t_rec = safe_get(metrics_df, best_transfer_name, 'recall')
        b_prec = safe_get(metrics_df, baseline_name, 'precision')
        t_prec = safe_get(metrics_df, best_transfer_name, 'precision')
        b_acc = safe_get(metrics_df, baseline_name, 'accuracy')
        t_acc = safe_get(metrics_df, best_transfer_name, 'accuracy')
        print(
            f"- Best transfer ({best_transfer_name}) vs baseline: "
            f"Accuracy {t_acc:.4f} ({t_acc-b_acc:+.4f}), Precision {t_prec:.4f} ({t_prec-b_prec:+.4f}), "
            f"Recall {t_rec:.4f} ({t_rec-b_rec:+.4f}), F1 {t_f1:.4f} ({t_f1-b_f1:+.4f})"
        )
    else:
        print('- Baseline or transfer rows are missing, so direct delta reporting was skipped.')

    if 'source_metrics_df' in globals() and not source_metrics_df.empty:
        print('- Source-level adaptability evidence is available in the previous table (metrics by source).')
    else:
        print('- Source-level adaptability could not be quantified due to missing/insufficient per-source class coverage.')

General Objective:
Develop and evaluate a multimodal fraud detection framework on local untouched holdout data.
Best evaluated multimodal setting by F1: transfer_local_only (F1=0.7273, Recall=0.9231, Precision=0.6000, Accuracy=0.8714)

Specific Objective 1 (improvement vs unimodal/baseline):
- baseline: ensemble F1 0.2105 vs best unimodal (metadata) F1 0.2667; uplift=-0.0561, recall uplift=+0.0000
- transfer_ablation: ensemble F1 0.6857 vs best unimodal (metadata) F1 0.8571; uplift=-0.1714, recall uplift=+0.0000
- transfer_local_only: ensemble F1 0.7273 vs best unimodal (metadata) F1 0.7200; uplift=+0.0073, recall uplift=+0.2308

Specific Objective 2 (adaptation to actual local datasets):
- Holdout IDs evaluated: 70
- Experiments loaded from actual run outputs:
  - baseline: overlap=70, label_mismatch=0, file=C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\local\local_baseline_holdout_predictions.csv
  - transfer_ablation: overlap=70, label_mismatch=0, file=C:\Us

In [12]:
if not per_exp_scored:
    print('Confusion matrices unavailable until holdout-scored predictions are loaded.')
else:
    n = len(per_exp_scored)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, (name, scored) in zip(axes, per_exp_scored.items()):
        cm = confusion_matrix(scored['fraud_label'], scored['pred'], labels=[0, 1])
        im = ax.imshow(cm, cmap='Blues')
        ax.set_title(f'Confusion Matrix\n{name}')
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred 0', 'Pred 1'])
        ax.set_yticklabels(['True 0', 'True 1'])
        for i in range(2):
            for j in range(2):
                ax.text(j, i, int(cm[i, j]), ha='center', va='center', color='black')

    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.04)
    plt.tight_layout()
    plt.show()

Confusion matrices unavailable until holdout-scored predictions are loaded.


In [13]:
if not per_exp_scored:
    print('ROC/PR curves unavailable until holdout-scored predictions are loaded.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for name, scored in per_exp_scored.items():
        RocCurveDisplay.from_predictions(
            scored['fraud_label'],
            scored['score'],
            name=name,
            ax=axes[0],
        )
        PrecisionRecallDisplay.from_predictions(
            scored['fraud_label'],
            scored['score'],
            name=name,
            ax=axes[1],
        )

    axes[0].set_title('ROC Curves (Untouched Holdout)')
    axes[1].set_title('Precision-Recall Curves (Untouched Holdout)')
    plt.tight_layout()
    plt.show()

ROC/PR curves unavailable until holdout-scored predictions are loaded.


In [23]:
if not per_exp_scored:
    print('Listing-level comparison unavailable until holdout-scored predictions are loaded.')
    comparison = holdout.copy()
    actual_summary = pd.DataFrame({
        'metric': ['total_holdout', 'actual_fraud', 'actual_legit'],
        'value': [
            len(holdout),
            int(holdout['fraud_label'].sum()),
            int((holdout['fraud_label'] == 0).sum()),
        ],
    })
    display(actual_summary)
else:
    # Build listing-level comparison table across experiments
    comparison = holdout.copy()

    for ex in experiments:
        name = ex['name']
        d = ex['df'][['product_id', 'score', 'pred']].copy()
        d = d.rename(columns={'score': f'{name}_score', 'pred': f'{name}_pred'})
        comparison = comparison.merge(d, on='product_id', how='left')

    pred_cols = [c for c in comparison.columns if c.endswith('_pred')]
    if pred_cols:
        comparison['num_models_predict_fraud'] = comparison[pred_cols].sum(axis=1, min_count=1)

    for ex in experiments:
        pcol = f"{ex['name']}_pred"
        if pcol in comparison.columns:
            comparison[f"{ex['name']}_correct"] = (comparison[pcol] == comparison['fraud_label']).astype('Int64')

    # Summary of actual outcomes
    actual_summary = pd.DataFrame({
        'metric': ['total_holdout', 'actual_fraud', 'actual_legit'],
        'value': [
            len(comparison),
            int(comparison['fraud_label'].sum()),
            int((comparison['fraud_label'] == 0).sum()),
        ],
    })

    actual_summary

In [24]:
if 'comparison' not in globals() or comparison.empty or not any(col.endswith('_pred') for col in comparison.columns):
    print('Error-focused listing views unavailable until holdout-scored predictions are loaded.')
else:
    # Error-focused views
    false_negatives = comparison[comparison['fraud_label'] == 1].copy()
    for ex in experiments:
        pcol = f"{ex['name']}_pred"
        if pcol in false_negatives.columns:
            false_negatives[f"{ex['name']}_missed_fraud"] = (false_negatives[pcol] == 0).astype(int)

    print('Fraud cases in holdout:')
    display(false_negatives[['product_id', 'fraud_label'] + [c for c in false_negatives.columns if c.endswith('_pred')]])

    print('Listing-level comparison (first 25 rows):')
    display(comparison.head(25))

    # Save a consolidated comparison CSV for reporting
    out_csv = ROOT / 'predictions' / 'transfer' / 'holdout_comparison_baseline_vs_transfer.csv'
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(out_csv, index=False)
    print(f'Saved consolidated listing-level comparison to: {out_csv}')

Fraud cases in holdout:


,product_id,fraud_label,baseline_pred,transfer_local_only_pred,transfer_ablation_pred
11,900060,1,0,1,1
16,900087,1,0,1,1
17,900088,1,0,1,1
24,900108,1,0,1,1
25,900112,1,0,1,1
27,900120,1,0,1,1
29,900125,1,0,1,1
51,900250,1,0,0,0
56,900282,1,0,1,1
59,900300,1,0,1,1


Listing-level comparison (first 25 rows):


,product_id,fraud_label,source,baseline_score,baseline_pred,transfer_local_only_score,transfer_local_only_pred,transfer_ablation_score,transfer_ablation_pred,num_models_predict_fraud,baseline_correct,transfer_local_only_correct,transfer_ablation_correct
0,900008,0,local1,0.2577,0,0.2837,0,0.2524,0,0,1,1,1
1,900009,0,local1,0.2873,0,0.2595,0,0.2537,0,0,1,1,1
2,900014,0,local1,0.2949,0,0.2592,0,0.2558,0,0,1,1,1
3,900018,0,local1,0.2736,0,0.2574,0,0.2229,0,0,1,1,1
4,900021,0,local1,0.3466,0,0.2779,0,0.2347,0,0,1,1,1
5,900022,0,local1,0.2690,0,0.2771,0,0.2310,0,0,1,1,1
6,900038,0,local1,0.2541,0,0.2665,0,0.2466,0,0,1,1,1
7,900052,0,local1,0.2927,0,0.2567,0,0.2468,0,0,1,1,1
8,900054,0,local1,0.2071,0,0.2617,0,0.2380,0,0,1,1,1
9,900056,0,local1,0.2228,0,0.3516,1,0.2216,0,1,1,0,1


Saved consolidated listing-level comparison to: C:\Users\micof\Documents\GitHub\e-commerce-fraud-detection\predictions\transfer\holdout_comparison_baseline_vs_transfer.csv


## How To Interpret

- Prioritize **F1** and **Recall** if your goal is catching more fraud.
- Check **Precision** and **False Positives** if review workload is costly.
- Use the listing-level CSV export to inspect which specific products changed between baseline and transfer models.

If you want an automated narrative paragraph for your report, add a final cell that reads `metrics_df` and prints a text summary of the winner by F1/Recall/ROC-AUC.